# CBT-18.95M Champion V3: Curriculum V5 SFT (Dual NVIDIA T4, 7 h session)

Continues `champion_final.pt` with the decontaminated Curriculum V5 SFT:
- Broad-distribution copy spans, article-stripped search arguments, verbatim calculator expressions.
- Parameterized nested schemas (database_query, create_event, file_manager, fetch_api) with short `<THINK>`.
- 20% long-addition replay; `_best.pt` is selected on generated held-out traces and requires exact held-out arithmetic.
- Benchmark free values are filtered with the benchmark's own `case_free_values`; the full training corpus is exported for auditing.

Budget: the SFT anneals fully within 6.0 h of wall clock. Setup, tests, evaluation and packaging use the rest of the 7 h.

Run all cells top to bottom. Attach the bundle as a Kaggle Dataset or upload `champion_v3_sft_bundle.zip`.

In [ ]:
# Cell 1: Hardware check (Turing sm_75, FP16 only, TF32 off)
import torch
print(f"[INFO] PyTorch {torch.__version__} | CUDA available: {torch.cuda.is_available()}")
assert torch.cuda.is_available(), "[FAIL] No GPU. Enable 'GPU T4 x2' in the notebook accelerator settings."
for i in range(torch.cuda.device_count()):
    p = torch.cuda.get_device_properties(i)
    print(f"[INFO] GPU {i}: {p.name} | {p.total_memory / 2**30:.1f} GB | sm_{p.major}{p.minor}")
if torch.cuda.device_count() < 2:
    print("[WARN] Fewer than 2 GPUs: throughput halves; the 6.0 h schedule still anneals fully.")
torch.backends.cuda.matmul.allow_tf32 = False
torch.backends.cudnn.allow_tf32 = False

In [ ]:
# Cell 2: Locate the bundle and copy it to a writable workspace (/kaggle/input is read-only)
import os, sys, glob, shutil, zipfile

WORK = "/kaggle/working/cbt" if os.path.isdir("/kaggle/working") else os.path.abspath("cbt_workspace")
MARKER = "champion_model/scaled_t4/sft_search.py"

def find_bundle_root():
    for base in [os.getcwd(), WORK]:
        if os.path.exists(os.path.join(base, MARKER)):
            return base
    for hit in glob.glob(f"/kaggle/input/**/{MARKER}", recursive=True):
        return hit[: -len(MARKER)].rstrip("/")
    for z in glob.glob("/kaggle/input/**/champion_v3_sft_bundle.zip", recursive=True) + glob.glob("champion_v3_sft_bundle.zip"):
        os.makedirs(WORK, exist_ok=True)
        zipfile.ZipFile(z).extractall(WORK)
        return WORK
    raise FileNotFoundError("[FAIL] Bundle not found. Attach champion_v3_sft_bundle.zip as a dataset or upload it.")

root = find_bundle_root()
if root.startswith("/kaggle/input"):
    shutil.copytree(root, WORK, dirs_exist_ok=True)
    root = WORK
WORK = os.path.abspath(root)
os.chdir(WORK)
if WORK not in sys.path:
    sys.path.insert(0, WORK)
base_ckpt = "champion_model/scaled_t4/checkpoints/champion_final.pt"
assert os.path.exists(base_ckpt), f"[FAIL] Missing base checkpoint {base_ckpt}"
print(f"[OK] Workspace: {WORK}")
print(f"[OK] Base checkpoint: {base_ckpt} ({os.path.getsize(base_ckpt) / 2**20:.1f} MB)")

In [ ]:
# Cell 3: Unit tests (architecture, masking, V5 generators, decontamination, dataset.py audit)
import subprocess, sys
r = subprocess.run([sys.executable, "-m", "unittest", "tests.test_scaled_t4"], capture_output=True, text=True)
print(r.stderr[-1500:])
assert r.returncode == 0, "[FAIL] Unit tests failed; do not launch the SFT."
print("[OK] Unit tests passed.")

In [ ]:
# Cell 4: Preflight: base checkpoint loads and a V5 stream sample passes the benchmark audit
import torch
from champion_model.scaled_t4.sft_search import build_benchmark_holdout, CurriculumV5Stream
from benchmarks.evaluate_champion import audit_contamination

ck = torch.load(base_ckpt, map_location="cpu", weights_only=False)
print(f"[OK] Base checkpoint step: {ck.get('step', ck.get('steps'))}")
prompts, values = build_benchmark_holdout()
stream = CurriculumV5Stream(seed=0, prompts=prompts, spans=values)
sample = "\n".join(i["full_text"] for i in stream.search(2000) + stream.tools(4000))
violations = audit_contamination(sample) + audit_contamination(open("champion_model/scaled_t4/dataset.py").read())
assert not violations, violations
print(f"[OK] Holdout: {len(prompts)} prompts, {len(values)} free values | sample audit: 0 violations ({stream.rejected} rejected)")

## Curriculum V5 SFT (6.0 h)

Watch for:
- `[HELDOUT]` every 2,000 steps: tool exact-call accuracy and held-out long-addition accuracy.
- `[WARN] Arithmetic regression`: a checkpoint failed held-out arithmetic and cannot become `_best.pt`.
- `[PROBE]` lines: qualitative check of tools and 9/25-digit carries.

The log is also written to `/kaggle/working/champion_v3_sft.log`.

In [ ]:
# Cell 5: Launch Curriculum V5 SFT -> champion_v3 (anneals fully within 6.0 h)
import os, subprocess, sys, time
env = os.environ.copy()
env["PYTHONUNBUFFERED"] = "1"
env["PYDEVD_DISABLE_FILE_VALIDATION"] = "1"
log_path = "/kaggle/working/champion_v3_sft.log" if os.path.isdir("/kaggle/working") else "champion_v3_sft.log"
cmd = ["bash", "champion_model/scaled_t4/train.sh", base_ckpt, "200000", "sft_v3"]
print("[INFO] " + " ".join(cmd), flush=True)
t0 = time.time()
with open(log_path, "w") as log:
    proc = subprocess.Popen(cmd, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in proc.stdout:
        log.write(line)
        log.flush()
        if "Warning" not in line and "warnings.warn" not in line:
            print(line, end="", flush=True)
    proc.wait()
print(f"[INFO] SFT exited with code {proc.returncode} after {(time.time() - t0) / 3600:.2f} h")
assert proc.returncode == 0, f"[FAIL] SFT failed; see {log_path}"

In [ ]:
# Cell 6: Select checkpoint and audit the exported V3 training corpus
import gzip, os, torch
from benchmarks.evaluate_champion import audit_contamination

ckdir = "champion_model/scaled_t4/checkpoints"
best, final = f"{ckdir}/champion_v3_final_best.pt", f"{ckdir}/champion_v3_final.pt"
for p in (best, final):
    if os.path.exists(p):
        c = torch.load(p, map_location="cpu", weights_only=False)
        print(f"[INFO] {p}: step {c.get('step')} | tools {c.get('heldout_acc')} | math {c.get('math_heldout_acc')}")
EVAL_CKPT = best if os.path.exists(best) else final
if EVAL_CKPT == final:
    print("[WARN] No _best.pt (no checkpoint kept exact held-out arithmetic). Evaluating final; check arithmetic before reporting.")

corpus = gzip.open(f"{ckdir}/champion_v3_final_corpus.txt.gz", "rt", encoding="utf-8").read()
violations = audit_contamination(corpus)
print(f"[INFO] V3 corpus: {corpus.count(chr(10)):,} training texts")
print("[PASS] V3 corpus audit: 0 violations" if not violations else f"[FAIL] {violations}")
print(f"[OK] Evaluating: {EVAL_CKPT}")

In [ ]:
# Cell 7: Fast benchmark suites (phrasing 26, OOD 18, unseen OOD 30)
# The full arithmetic sweep (20-70 digits x 30 samples) is long without KV caching; run it offline:
#   python benchmarks/evaluate_champion.py --ckpt <ckpt> --suite arithmetic --out-json results/champion_v3_arith.json
import subprocess, sys, os
out_dir = "/kaggle/working" if os.path.isdir("/kaggle/working") else "."
for suite in ("phrasing", "ood"):
    subprocess.run([sys.executable, "-u", "benchmarks/evaluate_champion.py", "--ckpt", EVAL_CKPT, "--suite", suite,
                    "--out-json", f"{out_dir}/champion_v3_{suite}.json"], check=True)
subprocess.run([sys.executable, "-u", "benchmarks/evaluate_unseen_ood.py", "--ckpt", EVAL_CKPT,
                "--out-json", f"{out_dir}/champion_v3_unseen_ood.json"], check=True)

In [ ]:
# Cell 8: Package outputs for download
import os, zipfile
out_dir = "/kaggle/working" if os.path.isdir("/kaggle/working") else "."
ckdir = "champion_model/scaled_t4/checkpoints"
files = [f"{ckdir}/{f}" for f in ("champion_v3_final_best.pt", "champion_v3_final.pt", "champion_v3_final_corpus.txt.gz")]
files += [f"{out_dir}/{f}" for f in ("champion_v3_sft.log", "champion_v3_phrasing.json", "champion_v3_ood.json", "champion_v3_unseen_ood.json")]
bundle = f"{out_dir}/champion_v3_outputs.zip"
with zipfile.ZipFile(bundle, "w", zipfile.ZIP_DEFLATED) as z:
    for f in files:
        if os.path.exists(f):
            z.write(f, os.path.basename(f))
            print(f"[OK] {f} ({os.path.getsize(f) / 2**20:.1f} MB)")
        else:
            print(f"[WARN] Missing {f}")
print(f"[OK] Download: {bundle} ({os.path.getsize(bundle) / 2**20:.1f} MB)")